In [3]:
def analyze_transactions(transactions):
    customers = {}
    transaction_ids = set()

    for t in transactions:
        if t["id"] in transaction_ids:
            raise ValueError("Duplicate transaction ID")

        transaction_ids.add(t["id"])

        customer = t["customer"]

        if customer not in customers:
            customers[customer] = {
                "total": 0,
                "categories": {}
            }

        if t["status"] == "success":
            amount = t["amount"]
            category = t["category"]

            customers[customer]["total"] += amount

            customers[customer]["categories"][category] = (
                customers[customer]["categories"].get(category, 0)
                + amount
            )

    report = []

    for customer, data in customers.items():
        categories = data["categories"]

        if categories:
            top_category = max(
                categories,
                key=categories.get
            )
        else:
            top_category = None

        report.append({
            "customer": customer,
            "total_spending": data["total"],
            "top_category": top_category
        })

    report.sort(
        key=lambda x: (-x["total_spending"], x["customer"])
    )

    for rank, customer in enumerate(report, start=1):
        customer["rank"] = rank

    return report


transactions = [
    {"id": 1, "customer": "Ahmed", "category": "Food",
     "amount": 500, "status": "success"},

    {"id": 2, "customer": "Ahmed", "category": "Travel",
     "amount": 200, "status": "failed"},

    {"id": 3, "customer": "Faizan", "category": "Food",
     "amount": 700, "status": "success"}
]

result = analyze_transactions(transactions)

for customer in result:
    print(customer)

{'customer': 'Faizan', 'total_spending': 700, 'top_category': 'Food', 'rank': 1}
{'customer': 'Ahmed', 'total_spending': 500, 'top_category': 'Food', 'rank': 2}


Task 02 | Longest Valid Window
Concepts: Sliding window, dictionaries
Problem: Write longest_unique_window(text) to find the longest contiguous substring with no repeated characters. Return the substring, start index, and length.
Requirements: Target O(n) time. On equal-length windows, select the earliest. Treat spaces and punctuation as characters.
Sample / expected behavior: Input: "abcabcbb" -> ("abc", 0, 3); "" -> ("", 0, 0).


In [4]:
def longest_unique_window(text):
    last_seen = {}

    start = 0
    max_length = 0
    best_start = 0

    for i, char in enumerate(text):

        if char in last_seen and last_seen[char] >= start:
            start = last_seen[char] + 1

        last_seen[char] = i

        length = i - start + 1

        if length > max_length:
            max_length = length
            best_start = start

    substring = text[best_start:best_start + max_length]

    return substring, best_start, max_length


print(longest_unique_window("abcabcbb"))
print(longest_unique_window(""))
print(longest_unique_window("pwwkew"))

('abc', 0, 3)
('', 0, 0)
('wke', 2, 3)


Task 03 | Recursive JSON Path Explorer
Concepts: Recursion, nested lists/dictionaries
Problem: Implement find_paths(data, target) that recursively searches nested dictionaries and lists and returns all paths whose value equals target.
Requirements: Use dot notation for dictionary keys and [index] for list positions. Preserve traversal order; do not treat bool True as numeric 1.
Sample / expected behavior: Input: {"a":[{"x":5},5],"b":{"y":5}}, target=5 -> ["a[0].x", "a[1]", "b.y"].


In [5]:
def find_paths(data, target):
    paths = []

    def search(value, path):

        if type(value) == type(target) and value == target:
            paths.append(path)

        elif isinstance(value, dict):
            for key, item in value.items():

                if path:
                    new_path = path + "." + str(key)
                else:
                    new_path = str(key)

                search(item, new_path)

        elif isinstance(value, list):
            for index, item in enumerate(value):
                new_path = path + "[" + str(index) + "]"

                search(item, new_path)

    search(data, "")

    return paths


data = {
    "a": [{"x": 5}, 5],
    "b": {"y": 5}
}

print(find_paths(data, 5))

['a[0].x', 'a[1]', 'b.y']


Task 04 | Dependency Execution Planner
Concepts: Graphs, sets, algorithms
Problem: Given tasks and prerequisite pairs (A,B) meaning A must finish before B, return a valid execution order.
Requirements: Detect cycles and raise ValueError with a clear message. For multiple available tasks, select alphabetically first. Reject unknown task names.
Sample / expected behavior: Tasks=["build","test","deploy"]; dependencies=[("build","test"),("test","deploy")] -> ["build","test","deploy"].


In [6]:
def execution_order(tasks, dependencies):
    graph = {task: [] for task in tasks}
    indegree = {task: 0 for task in tasks}

    for a, b in dependencies:
        if a not in graph or b not in graph:
            raise ValueError("Unknown task name")

        graph[a].append(b)
        indegree[b] += 1

    available = sorted(
        task for task in tasks if indegree[task] == 0
    )

    order = []

    while available:
        task = available.pop(0)
        order.append(task)

        for next_task in graph[task]:
            indegree[next_task] -= 1

            if indegree[next_task] == 0:
                available.append(next_task)
                available.sort()

    if len(order) != len(tasks):
        raise ValueError("Cycle detected in dependencies")

    return order


tasks = ["build", "test", "deploy"]

dependencies = [
    ("build", "test"),
    ("test", "deploy")
]

print(execution_order(tasks, dependencies))

['build', 'test', 'deploy']


Task 05 | Expression Validator and Evaluator
Concepts: Stacks, parsing, exception handling
Problem: Create a calculator for integer expressions containing +, -, *, /, parentheses, and whitespace. Apply standard operator precedence and parentheses.
Requirements: Do not use eval(), exec(), or third-party parsers. Handle unary minus, malformed expressions, and division by zero.
Sample / expected behavior: Input: "-(3 + 2) * 4 + 18 / 3" -> -14.0.


In [7]:
def calculate(expression):
    tokens = []
    number = ""

    for char in expression:
        if char.isdigit():
            number += char

        else:
            if number:
                tokens.append(int(number))
                number = ""

            if char in "+-*/()":
                tokens.append(char)
            elif char.isspace():
                continue
            else:
                raise ValueError("Invalid character")

    if number:
        tokens.append(int(number))

    tokens.append("END")
    index = 0

    def expression_value():
        nonlocal index

        result = term()

        while tokens[index] in ("+", "-"):
            operator = tokens[index]
            index += 1
            value = term()

            if operator == "+":
                result += value
            else:
                result -= value

        return result

    def term():
        nonlocal index

        result = factor()

        while tokens[index] in ("*", "/"):
            operator = tokens[index]
            index += 1
            value = factor()

            if operator == "*":
                result *= value
            else:
                if value == 0:
                    raise ZeroDivisionError("Cannot divide by zero")

                result /= value

        return result

    def factor():
        nonlocal index

        token = tokens[index]

        if token == "-":
            index += 1
            return -factor()

        if token == "+":
            index += 1
            return factor()

        if token == "(":
            index += 1
            result = expression_value()

            if tokens[index] != ")":
                raise ValueError("Missing closing parenthesis")

            index += 1
            return result

        if isinstance(token, int):
            index += 1
            return token

        raise ValueError("Invalid expression")

    result = expression_value()

    if tokens[index] != "END":
        raise ValueError("Invalid expression")

    return result


print(calculate("-(3 + 2) * 4 + 18 / 3"))

-14.0


Task 06 | Banking Ledger with OOP
Concepts: Classes, encapsulation, properties
Problem: Build Account and Bank classes. Support deposit, withdrawal, transfer, and account statements. Maintain immutable transaction records containing transaction ID, action, amount, and resulting balance.
Requirements: Reject nonpositive amounts, insufficient funds, and nonexistent accounts. A transfer must be atomic: both balances update or neither does.
Sample / expected behavior: Account A=1000, B=200; transfer 350 from A to B -> A=650, B=550.


In [8]:
from collections import namedtuple

Transaction = namedtuple(
    "Transaction",
    ["transaction_id", "action", "amount", "balance"]
)


class Account:
    def __init__(self, account_id, balance=0):
        self.account_id = account_id
        self.__balance = balance
        self.transactions = []

    @property
    def balance(self):
        return self.__balance

    def deposit(self, amount, transaction_id):
        if amount <= 0:
            raise ValueError("Amount must be positive")

        self.__balance += amount

        self.transactions.append(
            Transaction(
                transaction_id,
                "Deposit",
                amount,
                self.__balance
            )
        )

    def withdraw(self, amount, transaction_id):
        if amount <= 0:
            raise ValueError("Amount must be positive")

        if amount > self.__balance:
            raise ValueError("Insufficient funds")

        self.__balance -= amount

        self.transactions.append(
            Transaction(
                transaction_id,
                "Withdrawal",
                amount,
                self.__balance
            )
        )

    def statement(self):
        for transaction in self.transactions:
            print(transaction)


class Bank:
    def __init__(self):
        self.accounts = {}
        self.transaction_id = 1

    def create_account(self, account_id, balance=0):
        if account_id in self.accounts:
            raise ValueError("Account already exists")

        if balance < 0:
            raise ValueError("Invalid balance")

        self.accounts[account_id] = Account(
            account_id, balance
        )

    def deposit(self, account_id, amount):
        if account_id not in self.accounts:
            raise ValueError("Account not found")

        self.accounts[account_id].deposit(
            amount, self.transaction_id
        )

        self.transaction_id += 1

    def withdraw(self, account_id, amount):
        if account_id not in self.accounts:
            raise ValueError("Account not found")

        self.accounts[account_id].withdraw(
            amount, self.transaction_id
        )

        self.transaction_id += 1

    def transfer(self, from_id, to_id, amount):
        if from_id not in self.accounts:
            raise ValueError("Source account not found")

        if to_id not in self.accounts:
            raise ValueError("Destination account not found")

        if amount <= 0:
            raise ValueError("Amount must be positive")

        source = self.accounts[from_id]
        destination = self.accounts[to_id]

        if source.balance < amount:
            raise ValueError("Insufficient funds")

        transaction_id = self.transaction_id

        source.withdraw(amount, transaction_id)
        destination.deposit(amount, transaction_id)

        self.transaction_id += 1


bank = Bank()

bank.create_account("A", 1000)
bank.create_account("B", 200)

bank.transfer("A", "B", 350)

print("Account A:", bank.accounts["A"].balance)
print("Account B:", bank.accounts["B"].balance)

bank.accounts["A"].statement()

Account A: 650
Account B: 550
Transaction(transaction_id=1, action='Withdrawal', amount=350, balance=650)


Task 07 | Employee Payroll Engine
Concepts: Inheritance, polymorphism, abstract classes
Problem: Define abstract Employee with calculate_pay(). Implement SalariedEmployee, HourlyEmployee, and CommissionEmployee with different pay formulas. Generate a unified payroll summary.
Requirements: Hourly overtime above 40 hours pays 1.5x. Commission employees receive base + sales*rate. Prevent direct instantiation of Employee.
Sample / expected behavior: Hourly rate=20, hours=45 -> 950; Commission base=500, sales=4000, rate=.10 -> 900.


In [10]:
from abc import ABC, abstractmethod


class Employee(ABC):
    def __init__(self, name):
        self.name = name

    @abstractmethod
    def calculate_pay(self):
        pass


class SalariedEmployee(Employee):
    def __init__(self, name, monthly_salary):
        super().__init__(name)
        self.monthly_salary = monthly_salary

    def calculate_pay(self):
        return self.monthly_salary


class HourlyEmployee(Employee):
    def __init__(self, name, rate, hours):
        super().__init__(name)
        self.rate = rate
        self.hours = hours

    def calculate_pay(self):
        if self.hours <= 40:
            return self.rate * self.hours

        regular_pay = 40 * self.rate
        overtime_pay = (self.hours - 40) * self.rate * 1.5

        return regular_pay + overtime_pay


class CommissionEmployee(Employee):
    def __init__(self, name, base, sales, rate):
        super().__init__(name)
        self.base = base
        self.sales = sales
        self.rate = rate

    def calculate_pay(self):
        return self.base + self.sales * self.rate


def payroll_summary(employees):
    for employee in employees:
        print(
            employee.name,
            "Salary:",
            employee.calculate_pay()
        )


employees = [
    SalariedEmployee("Faizan", 50000),
    HourlyEmployee("Ahmed", 20, 45),
    CommissionEmployee("Usama", 500, 4000, 0.10)
]

payroll_summary(employees)

Faizan Salary: 50000
Ahmed Salary: 950.0
Usama Salary: 900.0


Task 08 | Cache Decorator with Expiration
Concepts: Decorators, closures, functools, time
Problem: Implement @ttl_cache(seconds) that caches results for repeated positional and keyword arguments and recomputes after expiration. Track cache hits and misses.
Requirements: Preserve function metadata with functools.wraps. Handle equivalent keyword argument order; document behavior for unhashable inputs. Use time.monotonic().
Sample / expected behavior: A call with x=2,y=3 followed by y=3,x=2 before expiry should return a cache hit.


In [11]:
import time
from functools import wraps


def ttl_cache(seconds):
    def decorator(func):
        cache = {}
        hits = 0
        misses = 0

        @wraps(func)
        def wrapper(*args, **kwargs):
            nonlocal hits, misses

            key = (args, tuple(sorted(kwargs.items())))
            current_time = time.monotonic()

            if key in cache:
                result, expiry = cache[key]

                if current_time < expiry:
                    hits += 1
                    return result

            misses += 1

            result = func(*args, **kwargs)

            cache[key] = (
                result,
                current_time + seconds
            )

            return result

        wrapper.cache_info = lambda: {
            "hits": hits,
            "misses": misses
        }

        return wrapper

    return decorator


@ttl_cache(5)
def add(x, y):
    print("Calculating...")
    return x + y


print(add(x=2, y=3))
print(add(y=3, x=2))
print(add.cache_info())

Calculating...
5
5
{'hits': 1, 'misses': 1}


Task 09 | Streaming Log Inspector
Concepts: Generators, iterators, file handling
Problem: Write a generator read_errors(path) that streams a log file line by line, yielding (line_number, timestamp, message) only for ERROR records. Build summarize_errors() to count errors by date.
Requirements: Do not load the whole file into memory. Skip malformed lines while tracking their count. Expected format: YYYY-MM-DD HH:MM:SS | LEVEL | message.
Sample / expected behavior: Input line: 2026-11-07 10:30:00 | ERROR | Connection lost -> (1,"2026-11-07 10:30:00","Connection lost").


In [13]:
from collections import defaultdict


def read_errors(path, stats=None):
    if stats is None:
        stats = {}

    stats["malformed"] = 0

    with open(path, "r") as file:
        for line_number, line in enumerate(file, start=1):
            parts = line.strip().split(" | ")

            if len(parts) != 3:
                stats["malformed"] += 1
                continue

            timestamp, level, message = parts

            if level == "ERROR":
                yield line_number, timestamp, message


def summarize_errors(path):
    error_count = defaultdict(int)
    stats = {}

    for line_number, timestamp, message in read_errors(path, stats):
        date = timestamp.split()[0]
        error_count[date] += 1

    print("Errors by date:")

    for date, count in error_count.items():
        print(date, count)

    print("Malformed lines:", stats["malformed"])


# Sample log file
with open("log.txt", "w") as file:
    file.write("2026-11-07 10:30:00 | ERROR | Connection lost\n")
    file.write("2026-11-07 10:35:00 | INFO | Server started\n")
    file.write("Invalid log line\n")
    file.write("2026-11-07 11:00:00 | ERROR | Database error\n")


for record in read_errors("log.txt"):
    print(record)

summarize_errors("log.txt")

(1, '2026-11-07 10:30:00', 'Connection lost')
(4, '2026-11-07 11:00:00', 'Database error')
Errors by date:
2026-11-07 2
Malformed lines: 1


Task 10 | CSV Inventory Reconciliation
Concepts: CSV, file handling, validation
Problem: Read inventory.csv (sku,name,quantity) and movements.csv (sku,type,units). Apply IN and OUT movements in order, then write updated_inventory.csv and rejected_movements.csv.
Requirements: Reject unknown SKUs, negative or zero units, invalid movement types, and any OUT causing negative stock. Continue processing after rejected rows.
Sample / expected behavior: SKU P01 starts at 10; OUT 4, OUT 9, IN 2 -> final 8; second movement rejected.


In [14]:
import csv


def reconcile_inventory():
    inventory = {}

    # Read inventory
    with open("inventory.csv", "r", newline="") as file:
        reader = csv.DictReader(file)

        for row in reader:
            sku = row["sku"]
            inventory[sku] = {
                "name": row["name"],
                "quantity": int(row["quantity"])
            }

    rejected = []

    # Read movements
    with open("movements.csv", "r", newline="") as file:
        reader = csv.DictReader(file)

        for row in reader:
            sku = row["sku"]
            movement_type = row["type"].upper()

            try:
                units = int(row["units"])

                if units <= 0:
                    raise ValueError

                if sku not in inventory:
                    raise ValueError

                if movement_type == "IN":
                    inventory[sku]["quantity"] += units

                elif movement_type == "OUT":
                    current_stock = inventory[sku]["quantity"]

                    if current_stock < units:
                        raise ValueError

                    inventory[sku]["quantity"] -= units

                else:
                    raise ValueError

            except ValueError:
                rejected.append(row)

    # Write updated inventory
    with open("updated_inventory.csv", "w", newline="") as file:
        writer = csv.DictWriter(
            file,
            fieldnames=["sku", "name", "quantity"]
        )

        writer.writeheader()

        for sku, data in inventory.items():
            writer.writerow({
                "sku": sku,
                "name": data["name"],
                "quantity": data["quantity"]
            })

    # Write rejected movements
    with open("rejected_movements.csv", "w", newline="") as file:
        writer = csv.DictWriter(
            file,
            fieldnames=["sku", "type", "units"]
        )

        writer.writeheader()
        writer.writerows(rejected)


# Sample inventory file
with open("inventory.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["sku", "name", "quantity"])
    writer.writerow(["P01", "Keyboard", 10])


# Sample movements file
with open("movements.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["sku", "type", "units"])
    writer.writerow(["P01", "OUT", 4])
    writer.writerow(["P01", "OUT", 9])
    writer.writerow(["P01", "IN", 2])


reconcile_inventory()

print("Inventory updated successfully")

Inventory updated successfully


Task 11 | Multi-File Student Gradebook
Concepts: File handling, dictionaries, exceptions
Problem: Merge students.csv (student_id,name) and scores.csv (student_id,subject,marks). Calculate per-student average, grade, and rank; export report.csv.
Requirements: Marks must be 0–100; reject unknown IDs and duplicate student-subject entries. Grades: A>=85, B>=70, C>=55, D>=40, otherwise F. Rank by average, then ID.
Sample / expected behavior: Student 101 with marks 90,80,85 -> average 85.0, grade A.


In [16]:
import csv


def get_grade(average):
    if average >= 85:
        return "A"
    elif average >= 70:
        return "B"
    elif average >= 55:
        return "C"
    elif average >= 40:
        return "D"
    else:
        return "F"


def create_report():
    students = {}

    # Read students
    with open("students.csv", "r", newline="") as file:
        reader = csv.DictReader(file)

        for row in reader:
            student_id = row["student_id"]

            students[student_id] = {
                "name": row["name"],
                "marks": []
            }

    seen = set()

    # Read scores
    with open("scores.csv", "r", newline="") as file:
        reader = csv.DictReader(file)

        for row in reader:
            student_id = row["student_id"]
            subject = row["subject"]

            try:
                marks = float(row["marks"])

                if marks < 0 or marks > 100:
                    raise ValueError

                if student_id not in students:
                    raise ValueError

                key = (student_id, subject)

                if key in seen:
                    raise ValueError

                seen.add(key)

                students[student_id]["marks"].append(marks)

            except ValueError:
                print("Rejected record:", row)

    report = []

    for student_id, data in students.items():
        marks = data["marks"]

        if marks:
            average = sum(marks) / len(marks)
        else:
            average = 0

        report.append({
            "student_id": student_id,
            "name": data["name"],
            "average": round(average, 2),
            "grade": get_grade(average)
        })

    # Sort by average descending, then ID
    report.sort(
        key=lambda x: (-x["average"], x["student_id"])
    )

    # Assign ranks
    for rank, student in enumerate(report, start=1):
        student["rank"] = rank

    # Write report
    with open("report.csv", "w", newline="") as file:
        fields = [
            "student_id",
            "name",
            "average",
            "grade",
            "rank"
        ]

        writer = csv.DictWriter(file, fieldnames=fields)

        writer.writeheader()
        writer.writerows(report)

    for student in report:
        print(student)


# Sample students file
with open("students.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["student_id", "name"])
    writer.writerow(["101", "Ahmed"])
    writer.writerow(["102", "Faizan"])


# Sample scores file
with open("scores.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["student_id", "subject", "marks"])
    writer.writerow(["101", "Python", 90])
    writer.writerow(["101", "SQL", 80])
    writer.writerow(["101", "OOP", 85])
    writer.writerow(["102", "Python", 75])
    writer.writerow(["102", "SQL", 70])


create_report()

{'student_id': '101', 'name': 'Ahmed', 'average': 85.0, 'grade': 'A', 'rank': 1}
{'student_id': '102', 'name': 'Faizan', 'average': 72.5, 'grade': 'B', 'rank': 2}


Task 12 | Custom Iterable Range Toolkit
Concepts: Iterators, generators, dunder methods
Problem: Create a ReversibleRange(start,stop,step) class supporting iteration, reversed iteration, len(), and membership tests without constructing a list.
Requirements: Match Python range stop-exclusive semantics for positive and negative nonzero steps. Implement __iter__, __reversed__, __len__, and __contains__.
Sample / expected behavior: ReversibleRange(2,11,3) -> [2,5,8]; reversed -> [8,5,2]; 5 in object -> True.


In [17]:
class ReversibleRange:
    def __init__(self, start, stop, step=1):
        if step == 0:
            raise ValueError("Step cannot be zero")

        self.start = start
        self.stop = stop
        self.step = step

    def __iter__(self):
        current = self.start

        if self.step > 0:
            while current < self.stop:
                yield current
                current += self.step

        else:
            while current > self.stop:
                yield current
                current += self.step

    def __reversed__(self):
        return reversed(range(
            self.start,
            self.stop,
            self.step
        ))

    def __len__(self):
        return len(range(
            self.start,
            self.stop,
            self.step
        ))

    def __contains__(self, value):
        return value in range(
            self.start,
            self.stop,
            self.step
        )


numbers = ReversibleRange(2, 11, 3)

print(list(numbers))
print(list(reversed(numbers)))
print(len(numbers))
print(5 in numbers)
print(6 in numbers)

[2, 5, 8]
[8, 5, 2]
3
True
False


Task 13 | Booking Conflict Scheduler
Concepts: Datetime, sorting, interval algorithms
Problem: Implement a room booking scheduler. Each booking has room, start datetime, end datetime, and booking ID. Accept bookings only when they do not overlap existing bookings for the same room.
Requirements: Use datetime objects; reject end<=start and duplicate IDs. Adjacent bookings are allowed. Return conflicts for rejected requests.
Sample / expected behavior: Room A: 09:00–10:00 and 10:00–11:00 accepted; 09:30–10:30 rejected.


In [18]:
from datetime import datetime


class BookingScheduler:
    def __init__(self):
        self.bookings = []
        self.booking_ids = set()

    def add_booking(self, room, start, end, booking_id):

        if booking_id in self.booking_ids:
            return "Rejected: Duplicate booking ID"

        if end <= start:
            return "Rejected: Invalid time"

        for booking in self.bookings:

            if booking["room"] == room:

                old_start = booking["start"]
                old_end = booking["end"]

                if start < old_end and end > old_start:
                    return "Rejected: Booking conflict"

        self.bookings.append({
            "room": room,
            "start": start,
            "end": end,
            "booking_id": booking_id
        })

        self.booking_ids.add(booking_id)

        self.bookings.sort(
            key=lambda x: (x["room"], x["start"])
        )

        return "Booking accepted"


scheduler = BookingScheduler()

start1 = datetime(2026, 11, 7, 9, 0)
end1 = datetime(2026, 11, 7, 10, 0)

start2 = datetime(2026, 11, 7, 10, 0)
end2 = datetime(2026, 11, 7, 11, 0)

start3 = datetime(2026, 11, 7, 9, 30)
end3 = datetime(2026, 11, 7, 10, 30)

print(scheduler.add_booking("A", start1, end1, 1))
print(scheduler.add_booking("A", start2, end2, 2))
print(scheduler.add_booking("A", start3, end3, 3))

Booking accepted
Booking accepted
Rejected: Booking conflict


Task 14 | Command-Line Contact Manager
Concepts: Modules, JSON, CRUD, validation
Problem: Build a menu-driven CLI contact manager with add, search, update, delete, list, and persistent JSON storage. Each contact has a unique ID, name, phone, and email.
Requirements: Use separate functions/modules for storage and business logic. Handle missing/corrupt JSON safely without silently deleting existing data. Search names case-insensitively.
Sample / expected behavior: Add ID=1, name="Ayesha"; restart application; search "aye" -> contact remains available.


In [ ]:
import json
import os


FILE_NAME = "contacts.json"


def load_contacts():
    if not os.path.exists(FILE_NAME):
        return {}

    try:
        with open(FILE_NAME, "r") as file:
            contacts = json.load(file)

            if not isinstance(contacts, dict):
                raise ValueError("Invalid contact data")

            return contacts

    except (json.JSONDecodeError, ValueError):
        raise ValueError(
            "Contacts file is corrupted. Data was not changed."
        )


def save_contacts(contacts):
    temp_file = FILE_NAME + ".tmp"

    with open(temp_file, "w") as file:
        json.dump(contacts, file, indent=4)

    os.replace(temp_file, FILE_NAME)


def add_contact(contacts):
    contact_id = input("Enter ID: ")

    if contact_id in contacts:
        print("ID already exists")
        return

    name = input("Enter name: ")
    phone = input("Enter phone: ")
    email = input("Enter email: ")

    contacts[contact_id] = {
        "name": name,
        "phone": phone,
        "email": email
    }

    save_contacts(contacts)
    print("Contact added")


def search_contact(contacts):
    name = input("Enter name to search: ").lower()

    for contact_id, contact in contacts.items():

        if name in contact["name"].lower():
            print(contact_id, contact)


def update_contact(contacts):
    contact_id = input("Enter ID: ")

    if contact_id not in contacts:
        print("Contact not found")
        return

    name = input("Enter new name: ")
    phone = input("Enter new phone: ")
    email = input("Enter new email: ")

    contacts[contact_id] = {
        "name": name,
        "phone": phone,
        "email": email
    }

    save_contacts(contacts)
    print("Contact updated")


def delete_contact(contacts):
    contact_id = input("Enter ID: ")

    if contact_id in contacts:
        del contacts[contact_id]
        save_contacts(contacts)
        print("Contact deleted")
    else:
        print("Contact not found")


def list_contacts(contacts):
    for contact_id, contact in contacts.items():
        print(contact_id, contact)


def main():
    try:
        contacts = load_contacts()
    except ValueError as error:
        print(error)
        return

    while True:
        print("\n1. Add Contact")
        print("2. Search Contact")
        print("3. Update Contact")
        print("4. Delete Contact")
        print("5. List Contacts")
        print("6. Exit")

        choice = input("Enter choice: ")

        if choice == "1":
            add_contact(contacts)

        elif choice == "2":
            search_contact(contacts)

        elif choice == "3":
            update_contact(contacts)

        elif choice == "4":
            delete_contact(contacts)

        elif choice == "5":
            list_contacts(contacts)

        elif choice == "6":
            break

        else:
            print("Invalid choice")


main()

Task 15 | Integrated Order Fulfillment System
Concepts: OOP, composition, files, exceptions, algorithms
Problem: Design Product, Cart, Order, and Inventory classes. Read a product catalog from JSON, build carts, apply percentage discounts, compute totals, and place orders with stock updates. Persist orders to JSON.
Requirements: Reject invalid quantities and out-of-stock purchases. Validate all items before changing inventory; roll back if order persistence fails. Generate unique order IDs. Include at least three unit tests using unittest.
Sample / expected behavior: Catalog: P1 price=100 stock=5; cart P1 x2 with 10% discount -> total 180, stock becomes 3 after successful order.


In [2]:
import json
import os
import uuid
import unittest


class Product:
    def __init__(self, product_id, name, price, stock):
        self.product_id = product_id
        self.name = name
        self.price = price
        self.stock = stock


class Cart:
    def __init__(self):
        self.items = {}

    def add_item(self, product_id, quantity):
        if quantity <= 0:
            raise ValueError("Quantity must be positive")

        self.items[product_id] = self.items.get(product_id, 0) + quantity


class Inventory:
    def __init__(self, products):
        self.products = products

    def check_stock(self, cart):
        for product_id, quantity in cart.items.items():
            if product_id not in self.products:
                raise ValueError("Product not found")

            if quantity > self.products[product_id].stock:
                raise ValueError("Not enough stock")

    def update_stock(self, cart):
        self.check_stock(cart)

        for product_id, quantity in cart.items.items():
            self.products[product_id].stock -= quantity


class Order:
    def __init__(self, cart, inventory, discount=0):
        self.order_id = str(uuid.uuid4())
        self.cart = cart
        self.inventory = inventory
        self.discount = discount

    def calculate_total(self):
        total = 0

        for product_id, quantity in self.cart.items.items():
            product = self.inventory.products[product_id]
            total += product.price * quantity

        return round(total * (1 - self.discount / 100), 2)

    def place_order(self):
        if not 0 <= self.discount <= 100:
            raise ValueError("Invalid discount")

        self.inventory.check_stock(self.cart)

        total = self.calculate_total()

        order_data = {
            "order_id": self.order_id,
            "items": self.cart.items,
            "discount": self.discount,
            "total": total
        }

        try:
            if os.path.exists("orders.json"):
                with open("orders.json", "r") as file:
                    orders = json.load(file)
            else:
                orders = []

            orders.append(order_data)

            with open("orders_temp.json", "w") as file:
                json.dump(orders, file, indent=4)

            os.replace("orders_temp.json", "orders.json")

        except Exception:
            if os.path.exists("orders_temp.json"):
                os.remove("orders_temp.json")

            raise

        self.inventory.update_stock(self.cart)

        return order_data


# Sample Product Catalog
products = {
    "P1": Product("P1", "Keyboard", 100, 5)
}

inventory = Inventory(products)

cart = Cart()
cart.add_item("P1", 2)

order = Order(cart, inventory, discount=10)

result = order.place_order()

print("Order ID:", result["order_id"])
print("Total:", result["total"])
print("Remaining Stock:", products["P1"].stock)


# Unit Testing
class TestOrderSystem(unittest.TestCase):

    def setUp(self):
        product = Product("P1", "Keyboard", 100, 5)

        self.inventory = Inventory({"P1": product})
        self.cart = Cart()

    def test_calculate_total(self):
        self.cart.add_item("P1", 2)

        order = Order(self.cart, self.inventory, 10)

        self.assertEqual(order.calculate_total(), 180)

    def test_stock_update(self):
        self.cart.add_item("P1", 2)

        order = Order(self.cart, self.inventory)
        order.place_order()

        self.assertEqual(
            self.inventory.products["P1"].stock,
            3
        )

    def test_insufficient_stock(self):
        self.cart.add_item("P1", 10)

        order = Order(self.cart, self.inventory)

        with self.assertRaises(ValueError):
            order.place_order()


if __name__ == "__main__":
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

..

.
----------------------------------------------------------------------
Ran 3 tests in 0.021s

OK


Order ID: d68a056e-8ca9-4e11-b30d-2a92a214fc94
Total: 180.0
Remaining Stock: 3
